# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² Mental Health Survey Dataset using the `mlcroissant` library. The dataset contains demographic and mental health survey responses from residents in Kilifi County, Kenya.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review the available record sets as well as their `@id`, fields, and column structure for further exploration.

In [ ]:
# List all record sets and their fields using their @id

print("Available record sets (referenced by '@id'):")
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print("  Fields (by @id and name):")
    for field in record_set.fields:
        print(f"    - {field.id}: {field.name} (dataType={field.data_type})")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All entities are referenced by their `@id`.

In [ ]:
# Extract data from each record set into pandas DataFrames

# List all available record set @id strings
record_sets = [rset.id for rset in metadata.record_sets]

dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# For demonstration, print the columns of the main survey record set
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"Columns for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering by criteria, normalizing numerical fields, and grouping data. All field references use their `@id`.

In [ ]:
# Example EDA using the GAD-7 sum score field as a numeric metric
# Use the correct field @id from the overview for analysis, e.g., 'gad7_score'

# Adjust these IDs as needed to match your dataset structure:
main_record_set_id = record_sets[0]  # e.g., 'cr:RecordSet_mental_health_survey'
numeric_field_id = None
group_field_id = None

main_df = dataframes[main_record_set_id]

# Identify numeric field
for record_set in metadata.record_sets:
    if record_set.id == main_record_set_id:
        for field in record_set.fields:
            if field.data_type in ['Float', 'Integer', 'Number'] and 'gad' in field.id.lower():
                numeric_field_id = field.id
            if group_field_id is None and (field.data_type == 'Text' and ('gender' in field.id.lower() or 'education' in field.id.lower())):
                group_field_id = field.id

if numeric_field_id is None:
    print("No suitable numeric field (e.g. GAD-7 score) found. Please set 'numeric_field_id' to a numeric field '@id'.")
else:
    threshold = 5
    # Avoid KeyError if the field is missing
    if numeric_field_id in main_df.columns:
        filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped normalized data by {group_field_id}:")
            print(grouped_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        print(f"Field '{numeric_field_id}' not found in DataFrame columns: {main_df.columns.tolist()}")

## 5. Visualization
Visualize the distribution of the main numeric field (e.g., GAD-7 scores) and group differences (e.g., by gender or education).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only plot if the fields exist and are sufficient
if numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' (e.g., GAD-7 Score)")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group if group_field found
    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR² Mental Health Survey Kilifi dataset, including the review of its record sets, data fields, and quick exploratory statistics and visualizations using only `@id`-based references. For your own analyses, adjust field and group identifiers as needed by referencing the precise `@id` values found in the schema overview section.

**Key findings and next steps:**
- The dataset includes comprehensive demographic and mental health indicators.
- Basic filtering, normalization, and grouping are supported directly via `mlcroissant` and pandas.
- Use visualizations to discover distributions and group differences (e.g., for GAD-7 or other psychometric measures).
- Continue with more detailed statistical analysis as appropriate for your research goals.
